In [4]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.20.0


In [5]:
nome_arquivo = '../datasets/dataset2.csv' 
df = pd.read_csv(nome_arquivo, sep=',') 

df.columns = df.columns.str.replace('#', '').str.strip().str.lower()
df.rename(columns={'heart_rate': 'bpm', 'activityid': 'target'}, inplace=True)

# Remove o ruído
df = df[df['target'] != 'transient activities'].copy()

# Rótulo
le = LabelEncoder()
df['target'] = le.fit_transform(df['target']) 

# 0 = Repouso (lying, sitting, standing associados aos códigos 0, 1, 2)
# 1 = Atividade/Anormal (o restante)
df['target'] = df['target'].apply(lambda x: 0 if x <= 2 else 1)
df.dropna(inplace=True)

print(f"Dados carregados e limpos. Shape atual: {df.shape}")

Dados carregados e limpos. Shape atual: (1936481, 33)


In [6]:
# Para prever problemas de saúde
window_size = 5

# Criação de janelas deslizantes
df['rolling_mean'] = df['bpm'].rolling(window=window_size).mean()
df['rolling_std'] = df['bpm'].rolling(window=window_size).std()
df['diff'] = df['bpm'].diff()

df.dropna(inplace=True)

# Definição das variáveis independentes (X) e dependente (y)
features = ['bpm', 'rolling_mean', 'rolling_std', 'diff']
X = df[features].values
y = df['target'].values

print(f"Features extraídas. Total de amostras prontas: {X.shape[0]}")

Features extraídas. Total de amostras prontas: 1936477


In [7]:
# Separação Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dados escalonados com sucesso.")

Dados escalonados com sucesso.


In [8]:
# Construção do modelo Sequencial
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)), 
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid") # Saída probabilística (0 a 1)
])

# Compilação do modelo
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.Recall(name='recall')]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,433 (9.50 KB)

 Trainable params: 2,433 (9.50 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
class_weight_dict = {0: 3.0, 1: 1.0}

# O histórico guarda as métricas por época para gráficos depois
history = model.fit(
    X_train_scaled, 
    y_train, 
    epochs=15, 
    batch_size=32,
    validation_split=0.2, 
    class_weight=class_weight_dict,
    verbose=1
)

Epoch 1/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 48s 1ms/step - accuracy: 0.7949 - loss: 0.5969 - recall: 0.7669 - val_accuracy: 0.7967 - val_loss: 0.3979 - val_recall: 0.7694
Epoch 2/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 46s 1ms/step - accuracy: 0.7986 - loss: 0.5893 - recall: 0.7735 - val_accuracy: 0.8041 - val_loss: 0.3970 - val_recall: 0.7834
Epoch 3/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 46s 1ms/step - accuracy: 0.7992 - loss: 0.5879 - recall: 0.7744 - val_accuracy: 0.7931 - val_loss: 0.4013 - val_recall: 0.7601
Epoch 4/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 46s 1ms/step - accuracy: 0.7986 - loss: 0.5876 - recall: 0.7731 - val_accuracy: 0.7892 - val_loss: 0.4086 - val_recall: 0.7524
Epoch 5/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 46s 1ms/step - accuracy: 0.7993 - loss: 0.5871 - recall: 0.7745 - val_accuracy: 0.8123 - val_loss: 0.3916 - val_recall: 0.7990
Epoch 6/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 50s 1ms/step - accuracy: 0.7982 - loss: 0.5867 - recall: 0.7724 - val_accuracy: 0.8146 - val_loss: 0.

In [10]:
# Realiza as previsões no conjunto de teste
probs_ativ = model.predict(X_test_scaled)

# Converte P(Atividade) para P(Repouso)
probs_repouso = 1.0 - probs_ativ.flatten()

# Limiar de segurança para a área da saúde
THRESHOLD = 0.8

# Se P(Repouso) >= 0.8 classifica como 0, senão 1
preds_custom = np.where(probs_repouso >= THRESHOLD, 0, 1)

print("\n--- Resultados Finais da Avaliação ---")
print("\nMatriz de Confusão:")
print(confusion_matrix(y_test, preds_custom))

print("\nRelatório de Classificação (Threshold = 0.8):")
print(classification_report(y_test, preds_custom, target_names=['Repouso (0)', 'Atividade/Anormal (1)']))

18155/18155 ━━━━━━━━━━━━━━━━━━━━ 11s 597us/step

--- Resultados Finais da Avaliação ---

Matriz de Confusão:
[[ 74953  66024]
 [ 29587 410380]]

Relatório de Classificação (Threshold = 0.8):
                       precision    recall  f1-score   support

          Repouso (0)       0.72      0.53      0.61    140977
Atividade/Anormal (1)       0.86      0.93      0.90    439967

             accuracy                           0.84    580944
            macro avg       0.79      0.73      0.75    580944
         weighted avg       0.83      0.84      0.83    580944



In [11]:
# Cria a pasta caso não exista
os.makedirs("neuralNetwork/models", exist_ok=True)

# Salva o modelo treinado
model.save("neuralNetwork/models/clinical_integrated_model.keras")

# Salva o padronizador
joblib.dump(scaler, "neuralNetwork/models/scaler.pkl")

print("Modelo e Scaler salvos com sucesso. Prontos para inferência no banco de dados.")

Modelo e Scaler salvos com sucesso. Prontos para inferência no banco de dados.
